# 01 — System setup and active-space integrals

Build the Ti2O4 cluster, run restricted Hartree–Fock with PySCF, and extract
core-corrected one- and two-electron integrals for six progressively larger
active spaces. Results are saved to `data/integrals.npz` for downstream
notebooks.

**Active spaces:** (4e,4o), (8e,8o), (12e,12o), (16e,16o), (20e,20o), (24e,24o).
The (12e,12o) target captures the Ti 3d / O 2p frontier manifold relevant to
photocatalysis on anatase TiO2.

In [1]:
import os
import sys
import numpy as np
from pyscf import gto, scf, ao2mo
import warnings
warnings.filterwarnings("ignore")

# Load configuration
sys.path.insert(0, ".")
from config import load_config, geometry_string, ensure_dirs
cfg = load_config()
ensure_dirs(cfg)
print(f"Loaded config: {cfg['system']['name']}, basis={cfg['system']['basis']}")

Loaded config: Ti2O4, basis=def2-svp


## Build the cluster and run RHF

Coordinates correspond to a Ti2O4 fragment derived from anatase TiO2.
The def2-SVP basis is a reasonable compromise between accuracy and cost
for transition-metal benchmark calculations.

In [2]:
mol = gto.Mole()
mol.atom = geometry_string(cfg)
mol.basis = cfg["system"]["basis"]
mol.charge = cfg["system"]["charge"]
mol.spin = cfg["system"]["spin"]
mol.symmetry = False
mol.verbose = 1
mol.max_memory = 4000
mol.build()

mf = scf.RHF(mol)
mf.max_cycle = cfg["scf"]["max_cycle"]
mf.conv_tol = cfg["scf"]["conv_tol"]
ehf = mf.kernel()

mo_energy = mf.mo_energy
mo_occ = mf.mo_occ
homo_idx = int(np.where(mo_occ > 0)[0][-1])
lumo_idx = homo_idx + 1
gap_ev = (mo_energy[lumo_idx] - mo_energy[homo_idx]) * 27.2114

print(f"System: Ti2O4, {mol.nelectron} electrons, {mol.nao_nr()} AOs")
print(f"HF energy: {ehf:.8f} Ha (converged: {mf.converged})")
print(f"HF HOMO-LUMO gap: {gap_ev:.4f} eV")

System: Ti2O4, 76 electrons, 118 AOs
HF energy: -1994.65160116 Ha (converged: True)
HF HOMO-LUMO gap: 3.8484 eV


## Active-space integrals with frozen core

For each active space, occupied orbitals below the active window are folded
into a core energy and an effective one-electron operator using the standard
mean-field potential:

$$h_{pq}^{\text{eff}} = h_{pq} + \sum_{i \in \text{core}}\bigl[2(pq|ii) - (pi|iq)\bigr]$$

In [3]:
active_space_configs = [
    (a["nelec"], a["norb"], a["label"]) for a in cfg["active_spaces"]
]

h1e_ao = mol.intor("int1e_kin") + mol.intor("int1e_nuc")
mo_coeff = mf.mo_coeff
all_data = {}

for nelec, norb, label in active_space_configs:
    n_occ = nelec // 2
    n_virt = norb - n_occ
    active_mos = list(range(homo_idx - n_occ + 1, homo_idx + n_virt + 1))
    mo_active = mo_coeff[:, active_mos]

    h1e_mo = mo_active.T @ h1e_ao @ mo_active
    h2e_chem = ao2mo.kernel(mol, mo_active, compact=False).reshape(norb, norb, norb, norb)
    h2e_phys = np.transpose(h2e_chem, (0, 2, 1, 3))

    core_start = active_mos[0]
    core_mos = mo_coeff[:, :core_start]
    n_core = core_mos.shape[1]
    e_nuc = mol.energy_nuc()

    if n_core > 0:
        dm_core = 2.0 * core_mos @ core_mos.T
        h1e_core = np.einsum("ij,ji->", h1e_ao, dm_core)
        eri_core = ao2mo.kernel(mol, core_mos, compact=False).reshape(
            n_core, n_core, n_core, n_core)
        j_core = 2.0 * np.einsum("iijj->", eri_core) - np.einsum("ijji->", eri_core)
        e_core = e_nuc + h1e_core + 0.5 * j_core

        eri_ca = ao2mo.kernel(mol, (mo_active, mo_active, core_mos, core_mos),
                              compact=False).reshape(norb, norb, n_core, n_core)
        eri_ac = ao2mo.kernel(mol, (mo_active, core_mos, core_mos, mo_active),
                              compact=False).reshape(norb, n_core, n_core, norb)
        v_core = 2.0 * np.einsum("pqii->pq", eri_ca) - np.einsum("piiq->pq", eri_ac)
        h1e_eff = h1e_mo + v_core
    else:
        e_core = e_nuc
        h1e_eff = h1e_mo

    all_data[label] = {
        "nelec": nelec,
        "norb": norb,
        "e_core": e_core,
        "h1e_eff": h1e_eff,
        "h2e_phys": h2e_phys,
    }

    print(f"({nelec}e, {norb}o) [{label:>7}]: e_core = {e_core:.4f} Ha, "
          f"frozen e- = {2 * n_core}, spin-orb = {2 * norb}")

(4e, 4o) [  small]: e_core = -2604.2962 Ha, frozen e- = 72, spin-orb = 8
(8e, 8o) [ medium]: e_core = -2554.5928 Ha, frozen e- = 68, spin-orb = 16
(12e, 12o) [ target]: e_core = -2498.5183 Ha, frozen e- = 64, spin-orb = 24
(16e, 16o) [  large]: e_core = -2441.6582 Ha, frozen e- = 60, spin-orb = 32
(20e, 20o) [ xlarge]: e_core = -2381.8404 Ha, frozen e- = 56, spin-orb = 40
(24e, 24o) [xxlarge]: e_core = -2317.8844 Ha, frozen e- = 52, spin-orb = 48


## Save integrals for downstream notebooks

In [4]:
data_path = os.path.join(cfg["paths"]["data_dir"], "integrals.npz")
np.savez_compressed(
    data_path,
    ehf=ehf,
    homo_idx=homo_idx,
    n_total_electrons=mol.nelectron,
    n_total_aos=mol.nao_nr(),
    **{f"{label}_nelec": np.array(d["nelec"]) for label, d in all_data.items()},
    **{f"{label}_norb": np.array(d["norb"]) for label, d in all_data.items()},
    **{f"{label}_e_core": np.array(d["e_core"]) for label, d in all_data.items()},
    **{f"{label}_h1e_eff": d["h1e_eff"] for label, d in all_data.items()},
    **{f"{label}_h2e_phys": d["h2e_phys"] for label, d in all_data.items()},
)
print(f"saved {data_path}")

saved data/integrals.npz
